# 04 - Ablation and Comparative Analysis

**Purpose.** Assemble research tables and figures from saved results only (no retraining): Track A comparison, CNN-Transformer comparison, quantum vs classical controls, quantum ablations, feature-module and preprocessing ablations, patient-level CIs and paired tests. Missing or failed experiments are listed explicitly.

**Requires (inputs).** Outputs of notebooks 00-03 for the same MODE (attach them as inputs on Kaggle).

**Produces (outputs, under `/kaggle/working/rsna_ich_artifacts/<MODE>/`).** `analysis/final_*.csv`, `analysis/final_*.md`, `analysis/figures/*.png`, `analysis/experiment_status.csv`.

> Research software. Outputs are model predictions for a research study and **not** clinical diagnoses.
> Results in this notebook are only valid for the MODE they were produced in (`smoke` results check correctness and say nothing about performance).

In [ ]:
# ==== Setup: locate the project code and make `src` importable ====================================
# Works on Kaggle (project code attached as a Dataset, possibly nested after zip extraction) and locally.
import os, sys, importlib
from pathlib import Path

def find_project_root(max_depth: int = 5) -> Path:
    def ok(p: Path) -> bool:
        return (p / "src" / "__init__.py").is_file() and (p / "configs" / "data_config.yaml").is_file()
    cands = []
    if os.environ.get("RSNA_ICH_PROJECT_ROOT"):
        cands.append(Path(os.environ["RSNA_ICH_PROJECT_ROOT"]))
    cands += [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/RSNA_ICH_Research")]
    for c in cands:
        if ok(c):
            return c.resolve()
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.exists():
            continue
        stack = [(base, 0)]
        while stack:
            d, depth = stack.pop()
            if ok(d):
                return d.resolve()
            if depth < max_depth:
                try:
                    stack += [(p, depth + 1) for p in d.iterdir() if p.is_dir() and not p.name.startswith("stage_2_")]
                except OSError:
                    pass
    raise FileNotFoundError(
        "Project code not found. On Kaggle: 'Add Input' -> the dataset containing RSNA_ICH_Research "
        "(with src/ and configs/). Locally: run the notebook from RSNA_ICH_Research/notebooks or set "
        "the RSNA_ICH_PROJECT_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _m in ["src", "src.utils.context", "src.data", "src.models", "src.quantum", "src.training", "src.evaluation"]:
    importlib.import_module(_m)  # fails loudly here if a dependency is missing
import src
print(f"Project root: {PROJECT_ROOT}  |  src version {src.__version__}")

In [ ]:
# ==== Configuration ================================================================================
# MODE: "smoke" (correctness check, minutes) | "pilot" (representative subset) | "full" (largest feasible).
MODE = "smoke"
# Optional per-config overrides without editing YAML, e.g. {"transformer": {"train": {"epochs": 2}}}
OVERRIDES = {}
import json
if os.environ.get("RSNA_ICH_OVERRIDES_JSON"):          # test hook used by tools/run_notebooks_synthetic.py
    OVERRIDES = json.loads(os.environ["RSNA_ICH_OVERRIDES_JSON"])
MODE = os.environ.get("RSNA_ICH_MODE", MODE)

from src.utils.context import ProjectContext
ctx = ProjectContext(PROJECT_ROOT, MODE, OVERRIDES)
print(ctx.describe())
print("Installed packages:", ctx.runtime.packages)
ctx.save_run_info("04_analysis")

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src import LABELS
from src.utils.io import atomic_write_json, atomic_save_dataframe, read_json, dataframe_to_markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

In [ ]:
manifest, man_info = ctx.manifest()
an_dir, fig_dir = ctx.dir("analysis"), ctx.dir("analysis", "figures")
res = ctx.results()
print(f"Found {len(res)} experiment result files across working dir and attached inputs.")
N_BOOT = {"smoke": 50, "pilot": 300, "full": 1000}[MODE]

## 1. Planned vs executed experiments (nothing is filled in for missing runs)

In [ ]:
tc, hc, qc = ctx.cfg["transformer"], ctx.cfg["hybrid"], ctx.cfg["quantum"]
planned = [(f"A_{m['name']}", "A: standalone Transformer") for m in tc["models"]]
planned += [(f"A_{m['name']}", "A: optional Transformer") for m in tc.get("optional_models", []) if m["name"] in tc.get("run_optional", [])]
planned += [("B_cnn_transformer_hybrid", "G: CNN-Transformer hybrid"), (f"B_cnn_reference_{hc['cnn_reference']['name']}", "CNN reference")]
planned += [(f"B_hybrid_fusion_{f}", "feature-module ablation") for f in hc.get("fusion_ablation", [])]
grid = list(qc["primary"]) + (list(qc["ablations"]) if qc["run_ablations"] else [])
planned += [(f"C_{s['id']}_s{seed}", "C/B/D/E/F: heads on frozen features") for s in grid for seed in qc["seeds"]]
status = {r["experiment_id"]: r["status"] for _, r in res.iterrows()} if len(res) else {}
h_runs = [e for e in status if e.startswith("H_")]
planned += [(e, "H: single-window input") for e in h_runs] or [("H_<best>_brain3", "H: single-window input")]
st = pd.DataFrame([{"experiment": e, "role": role, "status": status.get(e, "NOT RUN / NOT FOUND")} for e, role in planned])
display(st["status"].value_counts().to_frame("count")); display(st)
atomic_save_dataframe(st, an_dir / "experiment_status.csv")

## 2. Image-model comparison (Tracks A, B, G, H) with patient-level bootstrap CIs on the held-out test set

In [ ]:
from src.evaluation.predictions import load_predictions, arrays, align
from src.evaluation.statistics import bootstrap_table, paired_bootstrap, holm
from src.evaluation.visualization import plot_forest
img = res[res["experiment_id"].str.match(r"^(A|B|H)_") & (res["status"] == "succeeded")] if len(res) else res
rows, preds = [], {}
for _, r in img.iterrows():
    d = Path(r["_dir"])
    df, _ = load_predictions(d / "predictions_test.parquet", man_info["manifest_hash"])
    preds[r["experiment_id"]] = df
    p, t = arrays(df)
    ci = bootstrap_table(t, p, df["group_id"].to_numpy(), ["macro_auc", "macro_ap", "auc:any"], n_boot=N_BOOT, seed=0).set_index("metric")
    per = pd.read_csv(d / "metrics_test_per_label.csv").set_index("label")["roc_auc"]
    rows.append({"model": r["experiment_id"], "weights": r["build"]["status"], "val_macro_auc": r["val_metrics"]["macro_roc_auc"],
                 "estimate": ci.loc["macro_auc", "estimate"], "ci_low": ci.loc["macro_auc", "ci_low"], "ci_high": ci.loc["macro_auc", "ci_high"],
                 "test_macro_ap": ci.loc["macro_ap", "estimate"], **{f"auc_{l}": per[l] for l in LABELS},
                 "test_macro_f1": r["test_metrics"]["macro_f1"], "test_hamming": r["test_metrics"]["hamming_loss"],
                 "test_brier": r["test_metrics"]["mean_brier"]})
image_table = pd.DataFrame(rows).sort_values("val_macro_auc", ascending=False) if rows else pd.DataFrame()
display(image_table.round(4))
if len(image_table):
    atomic_save_dataframe(image_table, an_dir / "final_image_models.csv")
    (an_dir / "final_image_models.md").write_text(dataframe_to_markdown(image_table), encoding="utf-8")
    plot_forest(image_table, "model", fig_dir / "forest_image_models.png", "Held-out test macro ROC-AUC (patient bootstrap 95% CI)"); plt.show()

## 3. Paired comparisons on identical test images (patient-cluster bootstrap, Holm-adjusted)
Reference = the best model by **validation** macro AUC.

In [ ]:
recs = []
if len(image_table) >= 2:
    ref = image_table.iloc[0]["model"]
    for o in image_table["model"].iloc[1:]:
        ids, al = align({"a": preds[ref], "b": preds[o]})
        pa, t = arrays(al["a"]); pb, _ = arrays(al["b"])
        recs.append({"comparison": f"{ref} - {o}", "n_images": len(ids),
                     **paired_bootstrap(t, pa, pb, al["a"]["group_id"].to_numpy(), "macro_auc", N_BOOT, 0)})
paired = pd.DataFrame(recs)
if len(paired):
    paired["p_holm"] = holm(paired["p_value"])
    atomic_save_dataframe(paired, an_dir / "final_paired_image_models.csv")
display(paired.round(4))

## 4. Quantum vs classical controls and quantum ablations (from notebook 03 runs)

In [ ]:
from src.evaluation.statistics import seed_summary, paired_seed_test
c_res = res[res["experiment_id"].str.startswith("C_")] if len(res) else res
if len(c_res):
    c_res = c_res.assign(head=c_res["experiment_id"].str.replace(r"_s\d+$", "", regex=True).str[2:],
                         seed=c_res["experiment_id"].str.extract(r"_s(\d+)$")[0].astype(int))
    ok = c_res[c_res["status"] == "succeeded"].copy()
    ok["test_macro_auc"] = ok["test_metrics"].apply(lambda m: m["macro_roc_auc"])
    ok["val_macro_auc"] = ok["val_metrics"].apply(lambda m: m["macro_roc_auc"])
    ok["params"] = ok["head_params"].apply(lambda m: m["total"])
    q_table = pd.DataFrame([{"head": h, **{f"test_{k}": v for k, v in seed_summary(g["test_macro_auc"]).items()},
                             "val_mean": g["val_macro_auc"].mean(), "params": g["params"].iloc[0],
                             "failed_runs": int((c_res[(c_res["head"] == h)]["status"] != "succeeded").sum())}
                            for h, g in ok.groupby("head", sort=False)])
    display(q_table.round(4)); atomic_save_dataframe(q_table, an_dir / "final_quantum_heads.csv")
    (an_dir / "final_quantum_heads.md").write_text(dataframe_to_markdown(q_table), encoding="utf-8")
    from src.evaluation.visualization import plot_seed_comparison
    plot_seed_comparison(ok, "head", "test_macro_auc", fig_dir / "quantum_heads_by_seed.png", "Per-seed held-out test macro AUC"); plt.show()
    tests = []
    for a, b, label in [("vqc_q4_l2_cyclic", "bottleneck_mlp", "C vs B (parameter-matched classical block)"),
                        ("vqc_q4_l2_cyclic", "bottleneck_identity", "C vs bottleneck only"),
                        ("vqc_q4_l2_cyclic", "linear_probe", "C vs linear probe (A-features)"),
                        ("vqc_q4_l2_cyclic", "vqc_q4_l2_noent", "D: entanglement vs none"),
                        ("vqc_q4_l4_cyclic", "vqc_q4_l2_cyclic", "E: depth 4 vs 2"), ("vqc_q4_l1_cyclic", "vqc_q4_l2_cyclic", "E: depth 1 vs 2"),
                        ("vqc_q8_l2_cyclic", "bottleneck_mlp_k8", "F: 8 qubits vs k=8 classical"), ("vqc_q2_l2_cyclic", "vqc_q4_l2_cyclic", "F: 2 vs 4 qubits")]:
        A, B = ok[ok["head"] == a], ok[ok["head"] == b]
        if len(A) and len(B):
            m = A.merge(B, on="seed", suffixes=("_a", "_b"))
            tests.append({"ablation": label, "a": a, "b": b, **paired_seed_test(m["test_macro_auc_a"], m["test_macro_auc_b"])})
        else:
            tests.append({"ablation": label, "a": a, "b": b, "n": 0, "note": "not run"})
    abl = pd.DataFrame(tests); display(abl.round(4)); atomic_save_dataframe(abl, an_dir / "final_quantum_ablations.csv")
else:
    print("No Track-C results found (attach notebook 03 output).")

## 5. Statistical method (for the report)
* **CIs:** patient-cluster percentile bootstrap (patients resampled with replacement, all their slices kept). Slices are not treated as independent.
* **Paired model comparisons:** paired patient-cluster bootstrap of the macro-AUC difference on identical test images; two-sided bootstrap p-value; Holm correction across comparisons.
* **Seed variability (Track C):** mean, SD and t-interval across training seeds; paired t-test / Wilcoxon across seeds (small n - interpret with caution).
* **Thresholds** were selected on validation (Youden J per label) and frozen before test evaluation.
* **Assumptions:** patients are independent; percentile intervals may be anti-conservative when few patients are positive for a label (epidural).